## Connect to data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
print(os.listdir("/content/drive/MyDrive/COMPS760/dataset"))

['clip_durations.tsv', 'validated.tsv', 'final_sample_600_per_group.csv']


## Load data

In [ ]:
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/COMPS760/dataset"
validated = pd.read_csv(f"{DATA_DIR}/validated.tsv", sep="\t", low_memory=False)
print("Total Rows:", len(validated))

Total Rows: 1901978


## Check the accent labels

In [ ]:
accent_counts = validated["accents"].value_counts(dropna=True)
print(accent_counts.head(30))

accents
United States English                                                       453243
England English                                                             159875
India and South Asia (India, Pakistan, Sri Lanka)                           112099
Canadian English                                                             77238
Australian English                                                           55922
Non native speaker|German English                                            53295
Southern African (South Africa, Zimbabwe, Namibia)                           26898
Scottish English                                                             20293
New Zealand English                                                          15761
Northern Irish                                                               10546
Irish English                                                                 9528
Filipino                                                                      5

In [ ]:
TARGET_KEYWORDS = ["Filipino", "Hong Kong", "Malaysian", "Southern African", "India and South Asia"]

for kw in TARGET_KEYWORDS:
    print(f"\nall accent chars that including '{kw}':")
    print(validated[validated["accents"].str.contains(kw, na=False)]["accents"].value_counts())


all accent chars that including 'Filipino':
accents
Filipino                                                                                                                                                                                                                                                                                                                                                                                           5714
United States English|Filipino                                                                                                                                                                                                                                                                                                                                                                      251
Canadian English|Filipino                                                                                                                                          

In [ ]:
NATIVE_KEYWORDS = ["United States","England","Australian","Canadian","Scottish","Irish","New Zealand","Welsh"]

for native_kw in NATIVE_KEYWORDS:
    print(f"\nAll accent labels containing '{native_kw}'")
    print(validated[validated["accents"].str.contains(native_kw, na=False)]["accents"].value_counts())


All accent labels containing 'United States'
accents
United States English                              453243
United States English|England English                3208
United States English|Midwestern|Low|Demure          1511
United States English|Transatlantic English          1303
United States English|Scandinavian                   1128
                                                    ...  
United States English|Lithuanian                        1
United States English|Slovenian                         1
United States English|New York|italian american         1
United States English|country                           1
United States English|american                          1
Name: count, Length: 257, dtype: int64

All accent labels containing 'England'
accents
England English                                                                                                         159875
United States English|England English                                                         

## Make sure clean accent group per speaker

In [ ]:
def get_primary_accent(accent_str):
    if pd.isna(accent_str):
        return None
    return accent_str.split("|")[0].strip()

validated["primary_accent"] = validated["accents"].map(get_primary_accent)

SELECTED_ACCENTS = ["Filipino","Hong Kong English","Malaysian English","Southern African (South Africa, Zimbabwe, Namibia)","India and South Asia (India, Pakistan, Sri Lanka)",]

selected = validated[validated["primary_accent"].isin(SELECTED_ACCENTS)].copy()
selected = selected.dropna(subset=["sentence_id", "sentence", "client_id", "accents"])

print("Selected clips:", len(selected))
print(selected["primary_accent"].value_counts())
assert selected["primary_accent"].nunique() == 5

Selected clips: 153710
primary_accent
India and South Asia (India, Pakistan, Sri Lanka)     112440
Southern African (South Africa, Zimbabwe, Namibia)     28247
Filipino                                                5736
Hong Kong English                                       4779
Malaysian English                                       2508
Name: count, dtype: int64


In [ ]:
# Welsh raw pool already < HK，skip, and Irish English got few group so skip as well
NATIVE_CANDIDATES = ["United States English", "England English","Australian English", "Canadian English","Scottish English", "New Zealand English",]

native_pool = validated[validated["primary_accent"].isin(NATIVE_CANDIDATES)].copy()
native_pool = native_pool.dropna(subset=["sentence_id", "sentence", "client_id", "accents"])

print(native_pool["primary_accent"].value_counts())

primary_accent
United States English    468936
England English          168061
Canadian English          77542
Australian English        56068
Scottish English          20360
New Zealand English       15763
Name: count, dtype: int64


## Get translations from CoVoST2

In [ ]:
%%bash
mkdir -p covost2 && cd covost2
for lang in de zh-CN ja id; do
  wget -q "https://dl.fbaipublicfiles.com/covost/covost_v2.en_${lang}.tsv.tar.gz"
  tar -xzf "covost_v2.en_${lang}.tsv.tar.gz"
done
ls -la

total 420508
drwxr-xr-x 2 root       root           4096 Aug 30 08:53 .
drwxr-xr-x 1 root       root           4096 Aug 30 08:53 ..
-rw-rw-r-- 1 1185200165 1185200165 83469402 Oct 14  2020 covost_v2.en_de.tsv
-rw-r--r-- 1 root       root       25779505 Oct 14  2020 covost_v2.en_de.tsv.tar.gz
-rw-rw-r-- 1 1185200165 1185200165 80725679 Oct 14  2020 covost_v2.en_id.tsv
-rw-r--r-- 1 root       root       22904065 Oct 14  2020 covost_v2.en_id.tsv.tar.gz
-rw-rw-r-- 1 1185200165 1185200165 94077720 Oct 14  2020 covost_v2.en_ja.tsv
-rw-r--r-- 1 root       root       26664247 Jan  7  2021 covost_v2.en_ja.tsv.tar.gz
-rw-rw-r-- 1 1185200165 1185200165 72676722 Oct 14  2020 covost_v2.en_zh-CN.tsv
-rw-r--r-- 1 root       root       24280932 Oct 14  2020 covost_v2.en_zh-CN.tsv.tar.gz


## Check the CoVoST2 files

In [ ]:
import pandas as pd
import glob

for f in glob.glob("covost2/covost_v2.en_*.tsv"):
    if f.endswith(".tar.gz"):
        continue
    df = pd.read_csv(f, sep="\t", nrows=3)
    print(f, "Col name:", df.columns.tolist())

covost2/covost_v2.en_ja.tsv Col name: ['path', 'translation', 'split']
covost2/covost_v2.en_id.tsv Col name: ['path', 'translation', 'split']
covost2/covost_v2.en_zh-CN.tsv Col name: ['path', 'translation', 'split']
covost2/covost_v2.en_de.tsv Col name: ['path', 'translation', 'split']


In [ ]:
print(df.head(2))

                           path                      translation split
0    common_voice_en_699711.mp3  Sie wird schon in Ordnung sein.  test
1  common_voice_en_18132047.mp3             Ende gut, alles gut.  test


## Recover English sentence text

In [ ]:
lookup = validated[["path", "sentence", "primary_accent"]].drop_duplicates("path")

CANDIDATE_LANGS = {"de": "German", "zh-CN": "Chinese", "ja": "Japanese", "id": "Indonesian"}

recovered_frames = {}
for lang_code, lang_name in CANDIDATE_LANGS.items():
    cv2 = pd.read_csv(f"covost2/covost_v2.en_{lang_code}.tsv", sep="\t")
    cv2_full = cv2.merge(lookup, on="path", how="left")

    match_rate = cv2_full["sentence"].notna().mean()
    print(f"{lang_name}: ratio of path match sentence = {match_rate:.1%}"
          f"（{cv2_full['sentence'].notna().sum()}/{len(cv2_full)}）")

    recovered_frames[lang_code] = cv2_full[cv2_full["sentence"].notna()].copy()

German: ratio of path match sentence = 99.6%（796576/799608）
Chinese: ratio of path match sentence = 99.6%（848088/851359）
Japanese: ratio of path match sentence = 99.6%（850428/853710）
Indonesian: ratio of path match sentence = 99.6%（849356/852547）


## Compare candidate target languages

In [ ]:
import re

def normalize(s):
    if pd.isna(s):
        return ""
    s = s.strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s

selected["sentence_norm"] = selected["sentence"].map(normalize)

results = []
for lang_code, rec_df in recovered_frames.items():
    rec_df["sentence_norm"] = rec_df["sentence"].map(normalize)
    rec_set = set(rec_df["sentence_norm"]) - {""}

    matched = selected[selected["sentence_norm"].isin(rec_set)]
    per_accent = matched.groupby("primary_accent")["sentence_id"].nunique()

    print(f"\n{CANDIDATE_LANGS[lang_code]} :")
    print("The sentence can matching with standard translation:", matched["sentence_id"].nunique(),
          f"/ {selected['sentence_id'].nunique()}")
    print(per_accent)

    row = {"lang": CANDIDATE_LANGS[lang_code], "matched": matched["sentence_id"].nunique()}
    row.update(per_accent.to_dict())
    results.append(row)

summary = pd.DataFrame(results).set_index("lang")
display(summary)


German :
The sentence can matching with standard translation: 27470 / 133681
primary_accent
Filipino                                               1961
Hong Kong English                                      1230
India and South Asia (India, Pakistan, Sri Lanka)     22881
Malaysian English                                      1172
Southern African (South Africa, Zimbabwe, Namibia)     5089
Name: sentence_id, dtype: int64

Chinese :
The sentence can matching with standard translation: 28487 / 133681
primary_accent
Filipino                                               1992
Hong Kong English                                      1237
India and South Asia (India, Pakistan, Sri Lanka)     23789
Malaysian English                                      1214
Southern African (South Africa, Zimbabwe, Namibia)     5118
Name: sentence_id, dtype: int64

Japanese :
The sentence can matching with standard translation: 28491 / 133681
primary_accent
Filipino                                              

,matched,Filipino,Hong Kong English,"India and South Asia (India, Pakistan, Sri Lanka)",Malaysian English,"Southern African (South Africa, Zimbabwe, Namibia)"
lang,,,,,,
German,27470,1961,1230,22881,1172,5089
Chinese,28487,1992,1237,23789,1214,5118
Japanese,28491,1992,1237,23793,1214,5118
Indonesian,28469,1992,1237,23781,1214,5108


In [ ]:
for lang_code, lang_name in CANDIDATE_LANGS.items():
    rec = recovered_frames[lang_code]
    target_matched = rec[rec["primary_accent"].isin(selected["primary_accent"].unique())]

    print(f"\n{lang_name} :")
    print(target_matched["primary_accent"].value_counts())
    print("5 target accent total match:", len(target_matched), "row")


German :
primary_accent
India and South Asia (India, Pakistan, Sri Lanka)     29614
Southern African (South Africa, Zimbabwe, Namibia)     5173
Filipino                                               1840
Hong Kong English                                      1225
Malaysian English                                      1025
Name: count, dtype: int64
5 target accent total match: 38877 row

Chinese :
primary_accent
India and South Asia (India, Pakistan, Sri Lanka)     31650
Southern African (South Africa, Zimbabwe, Namibia)     5425
Filipino                                               1897
Hong Kong English                                      1239
Malaysian English                                      1101
Name: count, dtype: int64
5 target accent total match: 41312 row

Japanese :
primary_accent
India and South Asia (India, Pakistan, Sri Lanka)     31660
Southern African (South Africa, Zimbabwe, Namibia)     5425
Filipino                                               1897
Hong Kong En

## Spot-check a few matches by hand

In [ ]:
sample = recovered_frames["de"][
    recovered_frames["de"]["primary_accent"].isin(selected["primary_accent"].unique())
].sample(5)
print(sample[["path", "sentence", "translation", "primary_accent"]])

                                path  \
92606   common_voice_en_17275056.mp3   
545707     common_voice_en_17743.mp3   
595404  common_voice_en_18653444.mp3   
512248  common_voice_en_19136999.mp3   
531956  common_voice_en_18479958.mp3   

                                                 sentence  \
92606                    Together, we can rule the world!   
545707                            Not unless we're blind.   
595404  She saw who the intruder was: it was her lover...   
512248  To read about the background to these events, ...   
531956                     She should have stuck with it.   

                                              translation  \
92606           Zusammen können wir die Welt beherrschen!   
545707               Nicht so lange wir nicht blind sind.   
595404  Sie sah, wer der Eindringling war: es war ihr ...   
512248  Informationen zum Hintergrund dieser Ereigniss...   
531956                    Sie hätte dabei bleiben sollen.   

                         

## Setting to Chinese, and match both group pools against real translations

In [ ]:
TARGET_LANG = "zh-CN"
cv2_zh = recovered_frames[TARGET_LANG]
if "sentence_norm" not in cv2_zh.columns:
    cv2_zh["sentence_norm"] = cv2_zh["sentence"].map(normalize)
zh_sentence_set = set(cv2_zh["sentence_norm"]) - {""}

# keep only clips whose sentence has a real Chinese translation (5 accent groups)
gt_subset = selected[selected["sentence_norm"].isin(zh_sentence_set)].copy()

# clips per accent group
sample_counts = gt_subset.groupby("primary_accent").size()
print("(1) Clips per accent group (with Chinese ground truth):")
print(sample_counts)
print("If we sample each group independently, N is capped at:", sample_counts.min())

# unique sentences per accent group
sentence_counts = gt_subset.groupby("primary_accent")["sentence_id"].nunique()
print("\n (2) Unique sentences per accent group :")
print(sentence_counts)

# how many sentences are shared across multiple accent groups?
sentence_accent_map = gt_subset.groupby("sentence_norm")["primary_accent"].nunique()

print("\n (3) Sentences shared across accent groups: ")
for k in [2, 3, 4, 5]:
    n = (sentence_accent_map >= k).sum()
    print(f"  Shared by at least {k} accent groups: {n} sentences")

shared_all5 = sentence_accent_map[sentence_accent_map == 5].index
preview = gt_subset[gt_subset["sentence_norm"].isin(shared_all5)][["sentence", "primary_accent"]].drop_duplicates("sentence")
print(f"\nSentences shared by all 5 groups: {len(shared_all5)}")
if len(shared_all5) > 0:
    print(preview.head(10))

(1) Clips per accent group (with Chinese ground truth):
primary_accent
Filipino                                               2037
Hong Kong English                                      1259
India and South Asia (India, Pakistan, Sri Lanka)     35759
Malaysian English                                      1284
Southern African (South Africa, Zimbabwe, Namibia)     5657
dtype: int64
If we sample each group independently, N is capped at: 1259

 (2) Unique sentences per accent group :
primary_accent
Filipino                                               1992
Hong Kong English                                      1237
India and South Asia (India, Pakistan, Sri Lanka)     23789
Malaysian English                                      1214
Southern African (South Africa, Zimbabwe, Namibia)     5118
Name: sentence_id, dtype: int64

 (3) Sentences shared across accent groups: 
  Shared by at least 2 accent groups: 4110 sentences
  Shared by at least 3 accent groups: 892 sentences
  Shared by at l

In [ ]:
# native candidates: same Chinese translation requirement
native_pool["sentence_norm"] = native_pool["sentence"].map(normalize)
native_matched = native_pool[native_pool["sentence_norm"].isin(zh_sentence_set)].copy()

print("Clips per native candidate (with Chinese ground truth):")
print(native_matched.groupby("primary_accent")["sentence_id"].nunique())

Clips per native candidate (with Chinese ground truth):
primary_accent
Australian English       23954
Canadian English         18904
England English          41533
New Zealand English       9740
Scottish English          8969
United States English    94965
Name: sentence_id, dtype: int64


## Filter by audio duration, and remove clips with impossible data

In [ ]:
durations = pd.read_csv(f"{DATA_DIR}/clip_durations.tsv", sep="\t")
print(durations.columns.tolist())
durations.head(3)

['clip', 'duration[ms]']


,clip,duration[ms]
0,common_voice_en_17254553.mp3,5376
1,common_voice_en_499068.mp3,3048
2,common_voice_en_645838.mp3,1968


In [ ]:
gt_subset_dur = gt_subset.merge(durations, left_on="path", right_on="clip", how="left")

missing = gt_subset_dur["duration[ms]"].isna().sum()
print(f"Clips with no duration info: {missing} / {len(gt_subset_dur)}")

print("\nDuration distribution (milliseconds):")
print(gt_subset_dur["duration[ms]"].describe())
print("\nPercentiles:")
print(gt_subset_dur["duration[ms]"].quantile([0.01, 0.05, 0.5, 0.95, 0.99]))

Clips with no duration info: 0 / 45996

Duration distribution (milliseconds):
count    45996.000000
mean      4646.100704
std       1783.804205
min        816.000000
25%       3264.000000
50%       4392.000000
75%       5784.000000
max      25176.000000
Name: duration[ms], dtype: float64

Percentiles:
0.01    1728.0
0.05    2232.0
0.50    4392.0
0.95    7896.0
0.99    9384.0
Name: duration[ms], dtype: float64


In [ ]:
# use the 1st-99th percentile of the full pool, not of each group separately
MIN_DURATION_MS = gt_subset_dur["duration[ms]"].quantile(0.01)  # ~1728ms
MAX_DURATION_MS = gt_subset_dur["duration[ms]"].quantile(0.99)  # ~9384ms

print(f"Keeping clips between {MIN_DURATION_MS:.0f} and {MAX_DURATION_MS:.0f} ms")

Keeping clips between 1728 and 9384 ms


In [ ]:
def clean_by_duration_and_outliers(df, min_ms, max_ms, label):
    """
    df: a clip-level dataframe with a 'path' column (not yet merged with `durations`).
    Merges in clip_durations.tsv, keeps clips within [min_ms, max_ms], then drops rows whose text is impossibly long for their audio (word count > 2x what 4 words/sec
    already a fast speaking rate could physically produce). Those are almost always parsing errors, not real recordings.
    """
    merged = df.merge(durations, left_on="path", right_on="clip", how="left")
    missing = merged["duration[ms]"].isna().sum()
    print(f"[{label}] clips with no duration info: {missing} / {len(merged)}")

    within_range = merged[
        (merged["duration[ms]"] >= min_ms) & (merged["duration[ms]"] <= max_ms)
    ].copy()
    print(f"[{label}] clips per group after duration filtering:")
    print(within_range.groupby("primary_accent").size())

    within_range["sentence_len_words"] = within_range["sentence"].str.split().str.len()
    within_range["max_plausible_words"] = (within_range["duration[ms]"] / 1000) * 4
    outliers = within_range[within_range["sentence_len_words"] > within_range["max_plausible_words"] * 2]
    print(f"[{label}] rows with an impossible duration-to-text ratio: {len(outliers)}")
    if len(outliers) > 0:
        print(outliers[["path", "duration[ms]", "sentence_len_words", "primary_accent"]])

    clean = within_range[~within_range["path"].isin(outliers["path"])].copy()
    print(f"[{label}] after cleaning: {len(clean)} / {len(merged)} rows kept")
    return clean

In [ ]:
gt_final_clean = clean_by_duration_and_outliers(gt_subset, MIN_DURATION_MS, MAX_DURATION_MS, "5 groups")

[5 groups] clips with no duration info: 0 / 45996
[5 groups] clips per group after duration filtering:
primary_accent
Filipino                                               2023
Hong Kong English                                      1168
India and South Asia (India, Pakistan, Sri Lanka)     35048
Malaysian English                                      1264
Southern African (South Africa, Zimbabwe, Namibia)     5585
dtype: int64
[5 groups] rows with an impossible duration-to-text ratio: 4
                               path  duration[ms]  sentence_len_words  \
17960  common_voice_en_18672170.mp3          7464                 209   
20164  common_voice_en_18712732.mp3          6936                 945   
25356  common_voice_en_18723655.mp3          3960                 308   
25360  common_voice_en_18723710.mp3          3864                 283   

                                          primary_accent  
17960  India and South Asia (India, Pakistan, Sri Lanka)  
20164  India and South A

In [ ]:
gt_native_clean = clean_by_duration_and_outliers(native_matched, MIN_DURATION_MS, MAX_DURATION_MS, "native")

[native] clips with no duration info: 0 / 388785
[native] clips per group after duration filtering:
primary_accent
Australian English        30367
Canadian English          24822
England English           85203
New Zealand English       11204
Scottish English           9718
United States English    220540
dtype: int64
[native] rows with an impossible duration-to-text ratio: 18
                                path  duration[ms]  sentence_len_words  \
42489   common_voice_en_18723328.mp3          3456                 612   
70626   common_voice_en_18717476.mp3          4056                 568   
92121   common_voice_en_18708418.mp3          6408                 212   
178199  common_voice_en_18672780.mp3          6576                  57   
197647  common_voice_en_17395339.mp3          2496                  23   
253524  common_voice_en_18724505.mp3          3672                 363   
287401  common_voice_en_18680830.mp3          4128                 163   
287915  common_voice_en_1868

## Check sentence length across groups

In [ ]:
gt_final_clean["sentence_len_chars"] = gt_final_clean["sentence"].str.len()

print(gt_final_clean.groupby("primary_accent")["sentence_len_chars"].describe())
print(gt_final_clean.groupby("primary_accent")["sentence_len_words"].describe())

                                                      count       mean  \
primary_accent                                                           
Filipino                                             2023.0  53.059318   
Hong Kong English                                    1168.0  43.210616   
India and South Asia (India, Pakistan, Sri Lanka)   35044.0  49.941873   
Malaysian English                                    1264.0  48.592563   
Southern African (South Africa, Zimbabwe, Namibia)   5585.0  50.953089   

                                                          std   min   25%  \
primary_accent                                                              
Filipino                                            21.952571   6.0  35.5   
Hong Kong English                                   19.334963  10.0  29.0   
India and South Asia (India, Pakistan, Sri Lanka)   22.197198   5.0  32.0   
Malaysian English                                   20.632538  12.0  33.0   
Southern African (S

## Check speaker cap

In [ ]:
def speaker_cap_report(df, cap, label, test_caps=(50, 75, 100, 150, 200), top_n=3):
    """
    Checks single-speaker dominance (shows the top few speakers per group), prints total clips available at a few cap levels for reference, then applies the final
    `cap` and returns (speaker_counts, max_share, capped_totals).
    """
    speaker_counts = df.groupby(["primary_accent", "client_id"]).size().reset_index(name="n_clips")

    max_share = speaker_counts.groupby("primary_accent")["n_clips"].apply(lambda x: x.max() / x.sum())
    print(f"[{label}] share of clips from the single biggest speaker, per group:")
    print(max_share)

    top_speakers = speaker_counts.sort_values("n_clips", ascending=False).groupby("primary_accent").head(top_n)
    print(f"\n[{label}] top {top_n} speakers per group:")
    print(top_speakers)

    print(f"\n[{label}] total clips available at different caps:")
    for c in test_caps:
        totals = speaker_counts.assign(capped=lambda d: d["n_clips"].clip(upper=c)).groupby("primary_accent")["capped"].sum()
        print(f"\ncap = {c}:")
        print(totals)

    capped_totals = speaker_counts.assign(
        capped=lambda d: d["n_clips"].clip(upper=cap)
    ).groupby("primary_accent")["capped"].sum()
    print(f"\n[{label}] final totals at cap={cap} (this sets the usable N ceiling):")
    print(capped_totals)

    return speaker_counts, max_share, capped_totals

In [ ]:
CAP = 100

speaker_counts, max_share, capped_totals = speaker_cap_report(gt_final_clean, CAP, "5 groups")

[5 groups] share of clips from the single biggest speaker, per group:
primary_accent
Filipino                                              0.092437
Hong Kong English                                     0.404966
India and South Asia (India, Pakistan, Sri Lanka)     0.086691
Malaysian English                                     0.119462
Southern African (South Africa, Zimbabwe, Namibia)    0.052999
Name: n_clips, dtype: float64

[5 groups] top 3 speakers per group:
                                         primary_accent  \
1151  India and South Asia (India, Pakistan, Sri Lanka)   
237   India and South Asia (India, Pakistan, Sri Lanka)   
1177  India and South Asia (India, Pakistan, Sri Lanka)   
109                                   Hong Kong English   
1506  Southern African (South Africa, Zimbabwe, Nami...   
1575  Southern African (South Africa, Zimbabwe, Nami...   
1569  Southern African (South Africa, Zimbabwe, Nami...   
117                                   Hong Kong English   
7

In [ ]:
native_speaker_counts, native_max_share, native_capped_totals = speaker_cap_report(gt_native_clean, CAP, "native")

[native] share of clips from the single biggest speaker, per group:
primary_accent
Australian English       0.185246
Canadian English         0.051338
England English          0.042276
New Zealand English      0.412799
Scottish English         0.505145
United States English    0.046402
Name: n_clips, dtype: float64

[native] top 3 speakers per group:
             primary_accent  \
6031  United States English   
5842  United States English   
225      Australian English   
2619       Scottish English   
2502    New Zealand English   
6976  United States English   
1971        England English   
1527        England English   
2126        England English   
95       Australian English   
514        Canadian English   
375      Australian English   
866        Canadian English   
877        Canadian English   
2493    New Zealand English   
2510    New Zealand English   
2708       Scottish English   
2702       Scottish English   

                                              client_id  

## Compare native candidates and lock in the final pick

In [ ]:
comparison = pd.DataFrame({
    "raw_pool": native_pool["primary_accent"].value_counts(),
    "zh_matched": native_matched.groupby("primary_accent")["sentence_id"].nunique(),
    "after_duration_clean": gt_native_clean.groupby("primary_accent").size(),
    "n_speakers": native_speaker_counts.groupby("primary_accent").size(),
    "max_speaker_share": native_max_share,
    "after_cap100": native_capped_totals,
})
print(comparison.sort_values("after_cap100", ascending=False))

                       raw_pool  zh_matched  after_duration_clean  n_speakers  \
primary_accent                                                                  
United States English    468936       94965                220531        4755   
England English          168061       41533                 85202        1485   
Canadian English          77542       18904                 24816         543   
Australian English        56068       23954                 30365         462   
New Zealand English       15763        9740                 11204         113   
Scottish English          20360        8969                  9718         114   

                       max_speaker_share  after_cap100  
primary_accent                                          
United States English           0.046402        126993  
England English                 0.042276         41957  
Canadian English                0.051338         16838  
Australian English              0.185246         14264  
New Zeala

In [ ]:
FINAL_NATIVE_GROUPS = ["United States English", "England English"]

gt_native_final_clean = gt_native_clean[gt_native_clean["primary_accent"].isin(FINAL_NATIVE_GROUPS)].copy()
print(gt_native_final_clean.groupby("primary_accent").size())

primary_accent
England English           85202
United States English    220531
dtype: int64


## Combine into one unified dataset

In [ ]:
gt_all_clean = pd.concat([gt_final_clean, gt_native_final_clean], ignore_index=True)

print(f"gt_all_clean: {len(gt_all_clean)} rows across {gt_all_clean['primary_accent'].nunique()} accent groups")
print(gt_all_clean.groupby("primary_accent").size())

gt_all_clean: 350817 rows across 7 accent groups
primary_accent
England English                                        85202
Filipino                                                2023
Hong Kong English                                       1168
India and South Asia (India, Pakistan, Sri Lanka)      35044
Malaysian English                                       1264
Southern African (South Africa, Zimbabwe, Namibia)      5585
United States English                                 220531
dtype: int64


## Apply the speaker cap, then draw the final N=600 sample

In [ ]:
RANDOM_SEED = 760
N_PER_GROUP = 600

# apply the cap at row level
gt_all_capped = (
    gt_all_clean
    .groupby(["primary_accent", "client_id"], group_keys=False)
    .apply(lambda g: g.sample(n=min(len(g), CAP), random_state=RANDOM_SEED))
    .reset_index(drop=True)
)
print("After applying the speaker cap (row-level, not just a projected count):")
print(gt_all_capped.groupby("primary_accent").size())

After applying the speaker cap (row-level, not just a projected count):
primary_accent
England English                                        41957
Filipino                                                1817
Hong Kong English                                        695
India and South Asia (India, Pakistan, Sri Lanka)      24094
Malaysian English                                       1204
Southern African (South Africa, Zimbabwe, Namibia)      4422
United States English                                 126993
dtype: int64


/tmp/ipykernel_2823/738423522.py:8: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=min(len(g), CAP), random_state=RANDOM_SEED))


In [ ]:
# draw N=600 sample per group
final_sample = (
    gt_all_capped
    .groupby("primary_accent", group_keys=False)
    .apply(lambda g: g.sample(n=N_PER_GROUP, random_state=RANDOM_SEED))
    .reset_index(drop=True)
)
print(f"Final sample: {len(final_sample)} rows, {N_PER_GROUP} per group, "
      f"{final_sample['primary_accent'].nunique()} accent groups")
print(final_sample.groupby("primary_accent").size())

Final sample: 4200 rows, 600 per group, 7 accent groups
primary_accent
England English                                       600
Filipino                                              600
Hong Kong English                                     600
India and South Asia (India, Pakistan, Sri Lanka)     600
Malaysian English                                     600
Southern African (South Africa, Zimbabwe, Namibia)    600
United States English                                 600
dtype: int64


/tmp/ipykernel_2823/2121190846.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=N_PER_GROUP, random_state=RANDOM_SEED))


In [ ]:
final_sample["sentence_len_chars"] = final_sample["sentence"].str.len()
final_sample["sample_id"] = "spk_" + (final_sample.groupby("client_id").ngroup().astype(str).str.zfill(4))
final_sample = final_sample.drop(columns=["client_id"])

In [ ]:
cv2_zh_dedup = cv2_zh.drop_duplicates("sentence_norm")[["sentence_norm", "translation"]]
print(f"CoVoST2 before remove duolicate: {len(cv2_zh)}rows，after removing: {len(cv2_zh_dedup)}rows")

final_sample = final_sample.merge(cv2_zh_dedup, on="sentence_norm", how="left")
final_sample = final_sample.rename(columns={"translation": "reference_translation_zh"})

missing = final_sample["reference_translation_zh"].isna().sum()
print(f"rows missing reference translation: {missing}")  # should be 0

CoVoST2 before remove duolicate: 848088rows，after removing: 261223rows
rows missing reference translation: 0


In [ ]:
OUTPUT_PATH = f"{DATA_DIR}/final_sample_600_per_group.csv"
final_sample.to_csv(OUTPUT_PATH, index=False)
print(f"Saved: {OUTPUT_PATH}")

Saved: /content/drive/MyDrive/COMPS760/dataset/final_sample_600_per_group.csv


In [ ]:
print("client_id" in final_sample.columns)
print(final_sample[["sample_id"]].head())

False
  sample_id
0  spk_1329
1  spk_1385
2  spk_0337
3  spk_1445
4  spk_1337


In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/COMPS760/COMPS760_dataset_check.ipynb" --output result.html

[NbConvertApp] Converting notebook /content/drive/MyDrive/COMPS760/COMPS760_dataset_check.ipynb to html
[NbConvertApp] Writing 474745 bytes to /content/drive/MyDrive/COMPS760/result.html
